# SupplyChainSleuth_ModelTraining



## Startup cells

In [0]:
# Set environment variables for sagemaker_studio imports

import os
os.environ['DataZoneProjectId'] = '68unbxgzaaxeav'
os.environ['DataZoneDomainId'] = 'dzd-574rwuctwgq31j'
os.environ['DataZoneEnvironmentId'] = 'dt8hx568pmkt2f'
os.environ['DataZoneDomainRegion'] = 'us-east-1'

# create both a function and variable for metadata access
_resource_metadata = None

def _get_resource_metadata():
    global _resource_metadata
    if _resource_metadata is None:
        _resource_metadata = {
            "AdditionalMetadata": {
                "DataZoneProjectId": "68unbxgzaaxeav",
                "DataZoneDomainId": "dzd-574rwuctwgq31j",
                "DataZoneEnvironmentId": "dt8hx568pmkt2f",
                "DataZoneDomainRegion": "us-east-1",
            }
        }
    return _resource_metadata
metadata = _get_resource_metadata()

In [0]:
"""
Logging Configuration

Purpose:
--------
This sets up the logging framework for code executed in the user namespace.
"""

from typing import Optional


def _set_logging(log_dir: str, log_file: str, log_name: Optional[str] = None):
    import os
    import logging
    from logging.handlers import RotatingFileHandler

    level = logging.INFO
    max_bytes = 5 * 1024 * 1024
    backup_count = 5

    # fallback to /tmp dir on access, helpful for local dev setup
    try:
        os.makedirs(log_dir, exist_ok=True)
    except Exception:
        log_dir = "/tmp/kernels/"

    os.makedirs(log_dir, exist_ok=True)
    log_path = os.path.join(log_dir, log_file)

    logger = logging.getLogger() if not log_name else logging.getLogger(log_name)
    logger.handlers = []
    logger.setLevel(level)

    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")

    # Rotating file handler
    fh = RotatingFileHandler(filename=log_path, maxBytes=max_bytes, backupCount=backup_count, encoding="utf-8")
    fh.setFormatter(formatter)
    logger.addHandler(fh)

    logger.info(f"Logging initialized for {log_name}.")


_set_logging("/var/log/computeEnvironments/kernel/", "kernel.log")
_set_logging("/var/log/studio/data-notebook-kernel-server/", "metrics.log", "metrics")

In [0]:
import logging
from sagemaker_studio import ClientConfig, sqlutils, sparkutils, dataframeutils

logger = logging.getLogger(__name__)
logger.info("Initializing sparkutils")
spark = sparkutils.init()
logger.info("Finished initializing sparkutils")

In [0]:
def _reset_os_path():
    """
    Reset the process's working directory to handle mount timing issues.
    
    This function resolves a race condition where the Python process starts
    before the filesystem mount is complete, causing the process to reference
    old mount paths and inodes. By explicitly changing to the mounted directory
    (/home/sagemaker-user), we ensure the process uses the correct, up-to-date
    mount point.
    
    The function logs stat information (device ID and inode) before and after
    the directory change to verify that the working directory is properly
    updated to reference the new mount.
    
    Note:
        This is executed at module import time to ensure the fix is applied
        as early as possible in the kernel initialization process.
    """
    try:
        import os
        import logging

        logger = logging.getLogger(__name__)
        logger.info("---------Before------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)

        os.chdir("/home/sagemaker-user")

        logger.info("---------After------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)
    except Exception as e:
        logger.exception(f"Failed to reset working directory: {e}")

_reset_os_path()

## Notebook

In [0]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import warnings
warnings.filterwarnings('ignore')

In [0]:
import re
import json
import hashlib
import boto3
import pandas as pd

In [0]:
import sys
import subprocess

def initialize_env():
    packages = ['protobuf==6.31.1', 'transformers', 'torch', 'ipywidgets', 'newspaper3k']
    for package in packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "--upgrade", "--no-cache-dir"])
    print("Environment synchronized. Please restart your kernel now.")

initialize_env()

In [0]:
import torch
from transformers import BertTokenizer

region = "us-east-1"
table_name = "SupplyChainArticles"
dynamodb = boto3.resource('dynamodb', region_name=region)
table = dynamodb.Table(table_name)

response = table.scan()
df = pd.DataFrame(response['Items'])

def extract_relevant_context(text, keywords_list, max_tokens=450):
    sentences = re.split(r'(?<=[.!?]) +', text)
    relevant_sentences = []
    
    for sentence in sentences:
        if any(kw.lower() in sentence.lower() for kw in keywords_list):
            relevant_sentences.append(sentence)
    
    combined = " ".join(relevant_sentences)
    return combined[:2000] 

keywords = ["forced labor", "supply chain", "disruption", "factory", "sanctions", "strike"]
df['CleanText'] = df['Text'].apply(lambda x: extract_relevant_context(x, keywords))

print(f"{len(df)} articles ready for tokenization")

In [0]:
search_categories = {
    0: ['"forced labor"', '"child labor"', '"human trafficking"', '"modern slavery"', '"unpaid wages"', '"debt bondage"', '"withholding passports"', '"factory fire safety violation"', '"workplace fatality"', '"hazardous working conditions"', '"illegal mining"', '"sanctions evasion"', '"bribery scandal"'],
    1: ['"supply chain disruption"', '"environmental fine"', '"deforestation report"', '"carbon emissions violation"', '"regulatory probe"', '"safety audit failure"', '"labor strike"', '"union busting allegation"', '"product recall"', '"toxic waste leak"', '"greenwashing lawsuit"', '"supplier bankruptcy"'],
    2: ['"corporate social responsibility"', '"sustainability report"', '"quarterly earnings"', '"annual general meeting"', '"new warehouse opening"', '"business partnership"', '"charity donation"', '"stock market update"', '"ceo interview"', '"product launch"', '"renewable energy transition"', '"employee benefits update"']
}

master_keyword_list = [
    phrase.replace('"', '').lower() 
    for sublist in search_categories.values() 
    for phrase in sublist
]

df['CleanText'] = df['Text'].apply(lambda x: extract_relevant_context(x, master_keyword_list))

print(f"Updated CleanText for {len(df)} articles")

In [0]:
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.optim import AdamW 

model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizer.from_pretrained(model_name)
model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=3)

inputs = tokenizer(
    df['CleanText'].tolist(),
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt"
)

labels = torch.tensor(df['Label'].astype(int).values)

print(f"Model loaded and {len(df)} articles tokenized.")
print(f"Input Shape: {inputs['input_ids'].shape}")

In [0]:
import sagemaker
from sagemaker.pytorch import PyTorch

sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()

estimator = PyTorch(
    entry_point='train.py',       
    role=role,
    instance_count=1,
    instance_type='ml.g4dn.xlarge',
    framework_version='2.0',
    py_version='py310',
    output_path="s3://supplychain-sleuth-ml-assets-east1/archive/",
    sagemaker_session=sagemaker_session
)

estimator.fit()

## Shutdown cells

In [0]:
"""
Stop spark session and associated Athena Spark session
"""

from IPython import get_ipython as _get_ipython
_get_ipython().user_ns["spark"].stop()